# Генерація заголовків для новин

У цій лабораторній роботі Ви познайомитеся з великими мовними моделями на прикладі задачі генерації заголовків для текстів новинного характеру.
І хоча задача є доволі специфічною, ідентичні підходи можна використовувати для вирішення задач автоматичного реферування (стислого переказу) і перефразування текстів та діалогів, а також зворотного процесу: генерації текстів великої розмірності (книжок, статей, чи навіть програмного коду) за коротким описом.


## Kaggle

Локальна робота з великими мовними моделями потребує значних обчислювальних ресурсів. Саме тому для виконання цієї лабораторної роботи рекомендується використовувати сервіс Kaggle.

**Kaggle** - платформа для змагань з машинного навчання, де користувачі змагаються у створенні найкращих моделей для вирішення широкого спектру задач, запропонованих різноманітними компаніями та організаціями. Окрім того, платформа має схожий на Google Colab функціонал для інтерактивної роботи з Python, а також надає доступ до графічних прискорювачів з лімітом у 9 годин на один сеанс та 30 годин на тиждень загалом.

Дізнатися більше про Kaggle Notebooks можна [тут](https://www.kaggle.com/docs/notebooks#technical-specifications), також детальний опис процесу створення ноутбуку наведено [тут](https://www.datacamp.com/tutorial/tutorial-kaggle-datasets-tutorials-kaggle-notebooks).

## Набір даних

У лабораторній роботі використовується підготовлена та зменшена версія [українського датасету](https://huggingface.co/datasets/FIdo-AI/ua-news), що містить новини та їх заголовки.
Для завантаження датасету у якості вхідних даних для створеного ноутбуку необхідно:
- у секції **Input**, що знаходиться справа у меню налаштувань, натиснути на **Upload** та обрати **New Dataset**
- завантажити архів датасету та вказати `fido-dataset` у якості назви, файл буде автоматично розпаковано та додано до папки `/kaggle/input/`

## Підготовка середовища

Тренування моделі потребує мінімум 16 гігабайт VRAM, тому у налаштуваннях ноутбука (**Settings** -> **Accelerator**)  або у секції **Session options** необхідно обрати графічний прискорювач `GPU P100/T4/L4/A100`. Перевірити наявність графічного прискорювача у поточному сеансі можна за допомогою нижченаведеної команди:

In [1]:
# виводить інформацію про графічний прискорювач та його поточне навантаження
!nvidia-smi

Thu Nov 19 00:03:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%pip uninstall -y torch torchvision torchaudio torchtext xformers

%pip install --no-cache-dir --force-reinstall \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.14.1+cu130
Uninstalling torch-2.14.1+cu130:
  Successfully uninstalled torch-2.14.1+cu130
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 479.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 569.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 525.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 400.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 467.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 515.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

### Перезавантажити середовище: Run -> Restart & clear cell outputs

In [2]:
# Встановлення залежностей
%pip install -U transformers[sentencepiece]
%pip install -U datasets
%pip install -U evaluate
%pip install -U accelerate
%pip install -U peft
%pip install groq
%pip install bert-score

In [3]:
!python --version

Python 3.13.16


In [4]:
%pip uninstall -y torchao

Found existing installation: torchao 0.16.0
Uninstalling torchao-0.16.0:
  Successfully uninstalled torchao-0.16.0


### Знову перезавантажити середовище: Run -> Restart & clear cell outputs

In [1]:
# Імпорт бібліотек
import os
import time
import json

import torch
import wandb
import numpy as np
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import load_from_disk, Dataset
from evaluate import load
from groq import Groq

In [2]:
# Файли вхідних даних доступні лише для читання в каталозі `../input/`.
# Наприклад, виконавши цю команду, ви побачите список усіх файлів у каталозі введення

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Ви можете записати до 20 ГБ у поточний каталог (`/kaggle/working/`), який буде збережено як результуючі дані під час створення версії за допомогою "Save & Run All"
# Ви також можете записати тимчасові файли в `/kaggle/temp/`, але вони не будуть збережені поза поточним сеансом

In [10]:
# не потребується для Kaggle-середовища
!mkdir "/kaggle/working/"

In [ ]:
# не потребується для Kaggle-середовища
!unzip "fido-dataset.zip" -d "/kaggle/working/fido-dataset"

Archive:  fido-dataset.zip
  inflating: /kaggle/working/fido-dataset/cache-40a3a7a65adcc2f5.arrow  
  inflating: /kaggle/working/fido-dataset/cache-77a8597cc930c391.arrow  
  inflating: /kaggle/working/fido-dataset/cache-d4ffd818802c683f.arrow  
  inflating: /kaggle/working/fido-dataset/cache-f9115c0626f3a666.arrow  
  inflating: /kaggle/working/fido-dataset/data-00000-of-00001.arrow  
  inflating: /kaggle/working/fido-dataset/dataset_info.json  
  inflating: /kaggle/working/fido-dataset/state.json  


In [2]:
# визначення обчислювального пристрою для роботи, cuda використовується для сеансу з обраним графічним прискорювачем
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## 1. Підготовка даних

In [ ]:
# перенесення набору даних у робочий каталог (лише для Kaggle)
!cp -r "/kaggle/input/fido-dataset" "/kaggle/working/"

In [3]:
# функція для перевірки входження тексту у список
def check_uniques(example, uniques):
    return example['text'] not in uniques

# завантаження даних
dataset = load_from_disk("/kaggle/working/fido-dataset")

# поділ даних на набори для тренування та оцінки
train_test_data = dataset.train_test_split(test_size=0.4, seed=42)

# видалення дублікатів тренувальних даних з набору для оцінки
uniques = set(train_test_data['train']['text'])
train_test_data['test'] = train_test_data['test'].filter(check_uniques, fn_kwargs={"uniques": uniques})
# поділ набору для оцінки на тестовий та валідаційний
dev_test_data = train_test_data['test'].train_test_split(test_size=0.5, seed=42)
train_test_data, dev_test_data

(DatasetDict({
     train: Dataset({
         features: ['text', 'input_ids', 'attention_mask'],
         num_rows: 58531
     })
     test: Dataset({
         features: ['text', 'input_ids', 'attention_mask'],
         num_rows: 38761
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['text', 'input_ids', 'attention_mask'],
         num_rows: 19380
     })
     test: Dataset({
         features: ['text', 'input_ids', 'attention_mask'],
         num_rows: 19381
     })
 }))

In [4]:
train_data = train_test_data['train']
eval_data = dev_test_data['train']
test_data = dev_test_data['test']

# видалення дублікатів валідаційних даних з тестового набору
uniques = set(eval_data['text'])
test_data = test_data.filter(check_uniques, fn_kwargs={"uniques": uniques})

In [5]:
train_data, eval_data, test_data

(Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 58531
 }),
 Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 19380
 }),
 Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 19352
 }))

In [6]:
# для зменшення часу тренування використано лише 3000 прикладів для валідації та 500 для тестування
eval_data = Dataset.from_dict(eval_data[:3000])
test_data = Dataset.from_dict(test_data[:500])
train_data, eval_data, test_data

(Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 58531
 }),
 Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 3000
 }),
 Dataset({
     features: ['text', 'input_ids', 'attention_mask'],
     num_rows: 500
 }))

In [7]:
train_data[1005:1010]["text"]

["<|endoftext|>Текст: Компанія Xiaomi готує друге покоління ігрового смартфона Black Shark, який з'явиться у продажу вже навесні. Смартфон працює на процесорі Snapdragon 855 з 8 ГБ оперативної пам'яті і Android Pie, пише GSMArena . Ймовірно, смартфон отримає ім'я Black Shark 2, хоча назва поки офіційно не підтверджена. Гаджет повторює дизайн першого Black Shark і наступника Black Shark Helo - темний корпус з вузькими зеленими лініями. На шпигунському знімку пристрою видно, що подвійна камера буде розташована у верхньому лівому кутку. Схоже, що в ній ширококутний об'єктив буде розташований над звичайним. Втім, цілком можливо, що перед нами прототип і все ще може змінитися. Нещодавно Xiaomi представила 27-ватну швидку зарядку для Mi 9 - ймовірно, вона з'явиться і в Black Shark 2. Нещодавно стало відомо, що новий Black Shark з номером моделі SKW-A0 буде комплектуватися зарядним пристроєм MDY-10-EH, який також входить в комплект Mi 9. Новинка буде нести на борту від 8 Гбайт до 12 Гбайт опе

In [8]:
eval_data[45:50]["text"]

['<|endoftext|>Текст: Про це повідомляє Wionews. Землетрус стався в китайському регіоні Сіньцзян на заході країни. В результаті були пошкоджені дахи, сотні жителів були евакуйовані. Після землетрусу залізничне сполучення між столицею регіону Урумчі, Хотан та Аксу було затримано. Китай регулярно потерпає від землетрусів, особливо в його гірських західних й південно-західних регіонах.\n Назва: У Китаї стався потужний землетрус, є загиблі<|endoftext|>',
 '<|endoftext|>Текст: Про це повідомляє прес-служба МОЗ. У зв\'язку з цим, міністерство рекомендує якнайшвидше зробити щеплення проти дифтерії і дорослим, і дітям. "Якісна і безпечна вакцина є в усіх регіонах України, і надається безоплатно як для дітей, так і для дорослих", - йдеться у повідомленні. Зазначається, що рівень вакцинації проти дифтерії лишається низьким - лише половина українських дітей отримала захист третьою дозою вакцини АКДП у віці 18 міс. минулого року, і менше половини дорослих. Дифтерія небезпечна тим, що без негайного

In [9]:
test_data[55:60]["text"]

['<|endoftext|>Текст: Про це повідомляє CNN з посиланням на пресслужбу британського прем’єра. "У заяві Даунінг-стріт зазначено, що уряд має намір закінчити поточну сесію парламенту 8 жовтня. Нове засідання розпочнеться через шість днів, 14 жовтня, з виступу королеви, в якому буде визначено законодавчий порядок денний уряду", - йдеться у повідомленні. У заяві наголошується, що виступ королеви 14 жовтня дозволить уряду "викласти свої плани щодо системи охорони здоров’я, шкіл, боротьби зі злочинністю, інвестування в інфраструктуру та побудови сильної економіки". Тому, як зазначається, робота парламенту має бути призупинена протягом найкоротшого часу, щоб забезпечити всі необхідні матеріально-технічні приготування. Нагадаємо, Верховний суд Великої Британії визнав незаконним рішення прем\'єр-міністра Бориса Джонсона про припинення роботи парламенту на п’ять тижнів.\n Назва: Джонсон знову хоче призупинити роботу парламенту<|endoftext|>',
 '<|endoftext|>Текст: Призначення німця Фелікса Бриха,

## 2. Тренування мовної моделі

Для вирішення задачі генерації заголовків буде використано [мовну модель GPT-2](https://huggingface.co/benjamin/gpt2-large-wechsel-ukrainian), а саме її `large`-версію з 876 мільйонами параметрів, що була попередньо натренована на 40 гігабайтах текстових даних та адаптована для української мови.

GPT - сімейство генеративних мереж на базі архітектури Transformer, що містять тільки шари декодування (Decoder). Процес генерації тексту полягає у тому, що на кожному кроці генерації модель використовує увесь попередній текст (в тому числі і результати попередніх генерацій) для визначення найбільш імовірного наступного слова.

Детальний огляд архітектури Transformer представлено [тут](https://jalammar.github.io/illustrated-transformer/), а з GPT можна ознайомитися [тут](https://jalammar.github.io/illustrated-gpt2/).

<center>
    <figure>
        <img src="https://jalammar.github.io/images/xlnet/gpt-2-autoregression-2.gif" alt ="Concrete">
        <figcaption>
            <a href="https://jalammar.github.io/images/xlnet/gpt-2-autoregression-2.gif">Illustrated GPT-2 by Jay Alammar</a></figcaption>
    </figure>
</center>

<center>
    <figure>
        <img src="https://jalammar.github.io/images/gpt2/gpt2-self-attention-example-2.png" alt ="Concrete">
        <figcaption>
            <a href="https://jalammar.github.io/images/gpt2/gpt2-self-attention-example-2.png">Illustrated GPT-2 by Jay Alammar</a></figcaption>
    </figure>
</center>

<center>
    <figure>
        <img src="https://jalammar.github.io/images/gpt2/gpt2-output.png" alt ="Concrete">
        <figcaption>
            <a href="https://jalammar.github.io/images/gpt2/gpt2-output.png">Illustrated GPT-2 by Jay Alammar</a></figcaption>
    </figure>
</center>

### 2.1 Підготовка моделі

In [10]:
hf_model_name = "benjamin/gpt2-large-wechsel-ukrainian"

# завантаження токенізатора
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
tokenizer

GPT2Tokenizer(name_or_path='benjamin/gpt2-large-wechsel-ukrainian', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [11]:
# Зазвичай текстові дані мають різну довжину, але процедура тренування потребує
# однакової довжини вхідних даних для пакетної (batch) обробки - декілька прикладів
# об'єднуються в один пакет і обробляються одночасно.
# Якщо приклад занадто короткий, то до нього додається необхідна кількість `pad_token` токенів.
# Під час попереднього тренування GPT не використовувалися `pad_token`, так як
# текстовий корпус ділився на блоки одного розміру. В той же час, для нашої задачі
# тексти новин мають різну довжину, тому необхідно додати цей спеціальний токен.
# Також, щоб не проводити тренування нового токена, буде використано спеціальний токен
# <|endoftext|>, що позначає і початок (bos_token) і кінець (eos_token) послідовності.
tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.pad_token, tokenizer.eos_token, tokenizer.bos_token)
print(tokenizer.pad_token_id, tokenizer.eos_token_id, tokenizer.bos_token_id)

<|endoftext|> <|endoftext|> <|endoftext|>
0 0 0


In [12]:
# завантаження моделі
model = AutoModelForCausalLM.from_pretrained(hf_model_name)
# розміщення моделі на графічному прискорювачі
model.to(device)

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-large-wechsel-ukrainian
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [13]:
model.device

device(type='cuda', index=0)

### 2.2 Генерація

На кожному кроці генерації модель за допомогою функції Softmax будує розподіл ймовірностей, де кожен елемент представляє собою відповідне слово чи токен зі словника моделі.
Елемент з найбільшою ймовірністю обирається у якості найбільш ймовірного продовження тексту з урахуванням усіх попередніх слів. Цей процес вибору має декілька важливих параметрів, одним з яких є температура. Чим більша температура, тим менша відстань від наймовірнішого слова до найменш імовірного, і навпаки. На практиці це означає, що більша температура розширює набір елементів, з яких обирається наступне слово/токен, і модель більше "фантазує". В той же час, зменшення температури призводить до більшої детермінованості, але і генерації будуть однотипними.

<center>
    <figure>
        <img src="https://lena-voita.github.io/resources/lectures/lang_models/sampling/softmax_temperature-min.png" alt ="Concrete">
        <figcaption>
            <a href="https://lena-voita.github.io/resources/lectures/lang_models/sampling/softmax_temperature-min.png">NLP Course by Lena Voita</a></figcaption>
    </figure>
</center>



<center>
    <figure>
        <img src="https://lena-voita.github.io/resources/lectures/lang_models/sampling/temp_diversity_coherence-min.png" alt ="Concrete">
        <figcaption>
            <a href="https://lena-voita.github.io/resources/lectures/lang_models/sampling/temp_diversity_coherence-min.png">NLP Course by Lena Voita</a></figcaption>
    </figure>
</center>

In [14]:
# функція для генерації заголовків
def predict_title(model, inputs: list, postprocess=False, temperature=0.6, max_new_tokens=48) -> list:
    outputs = []
    # з no_grad() градієнти не будуть обчислюватися
    with torch.no_grad():
        for idx, row in enumerate(inputs):
            if (idx+1) % 100 == 0:
                print(f"Generated {idx+1} titles\n")
            # текст містить і новину і назву, необхідно видалити назву
            prompt = row.split("\n Назва:")[0] + "\n Назва:"
            batch = tokenizer([prompt], return_tensors='pt').to(device)
            output_tokens = model.generate(
                **batch,
                max_new_tokens=max_new_tokens, # визначає максимальну кількість згенерованих токенів
                do_sample=True, # семплювання замість жадібного декодування
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            # декодування згенерованих ідентифікаторів токенів у текст
            output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
            if postprocess:
                # генеративні моделі без інструкційного налаштування досить часто генерують поки не
                # вичерпано ліміт токенів шляхом повторення вже згенерованого тексту, тому необхідно
                # обрати лише перший результат.
                output = output.split("\n Назва:")[1]
            outputs.append(output.strip())
    return outputs

In [15]:
# генерація заголовків для перших 50 текстів з тестового набору
predicted_titles = predict_title(model, test_data[:50]['text'])

for row in predicted_titles:
    print(row, '\n\n')

Текст: Глава правління Укрпошти Ігор Смілянський сказав в інтерв'ю проекту KRYM, що його місячний оклад збільшився в 2018 році і становить 748 тис. грн. "Місячний оклад - 748 тис. грн. Більшість наших співробітників вважає, що я повинен отримувати 40 тис. Це 10 зарплат листонош", - зазначив він. Смілянський нагадав, що у нього три вищі освіти в американських вузах, і підкреслив, що може відзвітувати за всі свої доходи, на відміну від критикуючих його за високу зарплату депутатів, які "все життя в політиці, але у них будинок в Конча-Заспі і Mercedes". "Я розумію джерело цього. Я знаю, що більшості людей простіше сприймати депутата, який офіційно отримує $300, живе в Конча-Заспі в триповерховому будинку і їздить на Mercedes... Вони знають, що він вкрав, але скільки точно, не знають, а моя зарплата відома", - додав глава Укрпошти. За словами Смілянського, після його приходу в компанію вона почала прозоро проводити держзакупівлі. "Поки ще ніхто не зміг перебити ті ціни, за якими ми купуємо

Згенеровані заголовки здебільшого демонструють відсутність зв'язку з наданим текстом новини, що цілком природно для моделі без достатного тренування під нашу задачу.

### 2.3* Налаштування моделі під задачу

Повноцінне тренування моделі на обраному датасеті потребує як мінімум 20 гігабайт VRAM (P100 має лише 16) та багато часу: в залежності від гіперпараметрів (кількості епох, розміру пакетів та максимальної довжини вхідної послідовності) процедура тренування може потребувати одну або декілька діб.

Для прискорення тренування скористаємося однією доволі поширеною технікою PEFT (Parameter-Efficient Fine-Tuning).

In [16]:
# заморожування параметрів моделі, щоб для них не обчислювалися градієнти під час тренування
for param in model.parameters():
    param.requires_grad = False

### LoRA (Low-Rank Adaptation)

LoRA є одною з технік ефективного налаштування великих моделей без значних потреб у обчислювальних ресурсах.
Метод полягає у тому, що під час тренування параметри моделі не змінюються, а замість цього оновлюються лише адаптери.
Адаптери представляють собою декомпозицію матриці у вигляді добутку двох низькорангових матриць:

- якщо матриця параметрів `W` має розмір `DxD`, то її можна представити як добуток матриць `A` розмірністю `DxR` та `B` розмірністю `RxD`
- якщо ранг `R` набагато менший за `D`, то обчислення відбуватимуться значно швидше
- під час прямого поширення заморожені ваги моделі множаться на вхідний сигнал `X`, аналогічні дії виконуються і з адаптерами, а далі результати обох операцій додаються
- під час зворотного поширення оновлюються лише параметри адаптерів.

Детальніше ознайомитися з LoRA можна за [посиланням](https://www.datacamp.com/tutorial/mastering-low-rank-adaptation-lora-enhancing-large-language-models-for-efficient-adaptation).

<center>
    <figure>
        <img src="https://www.unite.ai/wp-content/uploads/2023/10/lora-animated.gif" alt ="Concrete">
        <figcaption>
            <a href="https://www.unite.ai/wp-content/uploads/2023/10/lora-animated.gif">LoRA by unite.ai</a></figcaption>
    </figure>
</center>

In [17]:
# конфігурація LoRa
# перелік усіх доступних параметрів: https://huggingface.co/docs/peft/package_reference/lora
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [18]:
# синтез нової моделі на базі підготовленої та конфігурації LoRA
model = get_peft_model(model, config)

model.device

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


device(type='cuda', index=0)

In [19]:
def print_trainable_parameters(model):
    """
    Обчислення кількості параметрів моделі, що будуть оновлюватися під час тренування
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

print_trainable_parameters(model)

trainable params: 2949120 || all params: 776979200 || trainable%: 0.3795622842928099


In [20]:
# очищення оперативної пам'яті (опціонально)
import gc
gc.collect()

3

#### Варіанти для тренування моделі

Варіант = Номер_у_списку % 4 + 1

|**№**| Max steps | Warmup steps | Eval steps | Save steps |
|-----|-----------|--------------|------------|------------|
|**1**| 1800      | 50           | 180        | 180        |
|**2**| 1900      | 100          | 190        | 190        |
|**3**| 2000      | 150          | 250        | 250        |
|**4**| 2100      | 200          | 300        | 300        |

In [21]:
# конфігурація тренера
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4, # розмір пакету для тренування
        per_device_eval_batch_size=4, # розмір пакету для валідації
        gradient_accumulation_steps=4, # https://discuss.huggingface.co/t/batch-size-vs-gradient-accumulation/5260/6
        warmup_steps=100, # перші N тренувальних кроків learning_rate буде поступово зростати до заданого значення, щоб уникнути упередженість через порядок прикладів
        max_steps=3000, # загальна кількість тренувальних кроків
        learning_rate=2e-4,
        logging_steps=25,
        output_dir='outputs_gpt2large_title_generation_lr2e04',
        do_eval=True,
        eval_strategy="steps",
        eval_steps=300, # валідація відбувається кожні N тренувальних кроків
        save_steps=300, # параметри моделі зберігаються кожні N тренувальних кроків
        fp16=True, # тренування зі змішаною точністю дозволяє збільшити швидкість процесу
        bf16=False,
        #auto_find_batch_size=True, знаходить оптимальний розмір пакету у випадку OOM помилок
        #use_cpu=True, причини помилок під час тренування набагато легше визначати перезапуском процедури на CPU
        report_to="wandb"
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

In [ ]:
# логування
os.environ["WANDB_PROJECT"] = "gpt2large_title_generation_lr2e04_step3000"
# зареєструйтесь на платформі wandb.ai та згенеруйте API-ключ для логування https://wandb.ai/authorize
os.environ["WANDB_API_KEY"] = "<YOUR_API_KEY>"
wandb.login()

In [25]:
# тренування
# повне тренування не є обов`язковим, але роботи без повного тренування отримають меншу кількість балів
model.config.use_cache = False
trainer.train()

wandb: Detected [groq] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
300,2.475913,2.373381
600,2.414339,2.355516
900,2.417624,2.346275


Step,Training Loss,Validation Loss
300,2.475913,2.373381
600,2.414339,2.355516
900,2.417624,2.346275
1200,2.398564,2.339028
1500,2.446948,2.334386
1800,2.438459,2.330809
2100,2.416568,2.327905
2400,2.407825,2.325542
2700,2.423175,2.324251
3000,2.400381,2.323901


TrainOutput(global_step=3000, training_loss=2.4312616996765137, metrics={'train_runtime': 5049.1734, 'train_samples_per_second': 9.507, 'train_steps_per_second': 0.594, 'total_flos': 6.943753244055552e+16, 'train_loss': 2.4312616996765137, 'epoch': 0.8200642383653386})

In [26]:
# збереження моделі після завершення тренування
torch.save(model.state_dict(), '/kaggle/working/weights_gpt2large_title_generation_lr2e04_step3000_lora.pt')
model = model.merge_and_unload()
model.save_pretrained("/kaggle/working/complete_model_gpt2large_title_generation_lr2e04_step3000")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# опціонально: завантаження натренованої моделі у Ваш репозиторій на платформі huggingface.co
tokenizer.push_to_hub("<HUGGINGFACE_REPO_PATH>", token="<YOUR_HUGGINGFACE_TOKEN>")
model.push_to_hub("<HUGGINGFACE_REPO_PATH>", token="<YOUR_HUGGINGFACE_TOKEN")

## 3. Тестування моделі

In [30]:
# завантаження моделі з контрольної точки (тільки якщо тренування моделі було завершено)
#adapter_checkpoint_path = "/kaggle/working/outputs_gpt2large_title_generation_lr2e04/checkpoint-3000/" для Kaggle
adapter_checkpoint_path = "/content/outputs_gpt2large_title_generation_lr2e04/checkpoint-3000/"

base_model = AutoModelForCausalLM.from_pretrained(
    hf_model_name,
    device_map=device,
)

model = PeftModel.from_pretrained(base_model, adapter_checkpoint_path, device_map=device)
model.to(device)

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: benjamin/gpt2-large-wechsel-ukrainian
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1280)
        (wpe): Embedding(1024, 1280)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-35): 36 x GPT2Block(
            (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=3840, nx=1280)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1280, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3840, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
   

In [ ]:
# завантаження натренованої моделі з репозиторія платформи huggingface.co
# (тільки якщо тренування моделі не було завершено)
tokenizer = AutoTokenizer.from_pretrained("NLPForUA/gpt2-large-uk-title-generation")
model = AutoModelForCausalLM.from_pretrained("NLPForUA/gpt2-large-uk-title-generation")

model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

#### Варіанти для генерації заголовків

Варіант = Номер_у_списку % 4 + 1

|**№**| Temperature |
|-----|-------------|
|**1**| 0.5         |
|**2**| 0.55        |
|**3**| 0.6         |
|**4**| 0.65        |

In [31]:
# генерація заголовків для перших 10 текстів з тестового набору
predicted_titles = predict_title(model, test_data[:10]['text'], temperature=0.5)

for row in predicted_titles:
    print(row, '\n\n')

Текст: Глава правління Укрпошти Ігор Смілянський сказав в інтерв'ю проекту KRYM, що його місячний оклад збільшився в 2018 році і становить 748 тис. грн. "Місячний оклад - 748 тис. грн. Більшість наших співробітників вважає, що я повинен отримувати 40 тис. Це 10 зарплат листонош", - зазначив він. Смілянський нагадав, що у нього три вищі освіти в американських вузах, і підкреслив, що може відзвітувати за всі свої доходи, на відміну від критикуючих його за високу зарплату депутатів, які "все життя в політиці, але у них будинок в Конча-Заспі і Mercedes". "Я розумію джерело цього. Я знаю, що більшості людей простіше сприймати депутата, який офіційно отримує $300, живе в Конча-Заспі в триповерховому будинку і їздить на Mercedes... Вони знають, що він вкрав, але скільки точно, не знають, а моя зарплата відома", - додав глава Укрпошти. За словами Смілянського, після його приходу в компанію вона почала прозоро проводити держзакупівлі. "Поки ще ніхто не зміг перебити ті ціни, за якими ми купуємо

In [32]:
# генерація заголовків для усього тестового набору
predicted_titles = predict_title(model, test_data['text'], postprocess=True, temperature=0.5)

for idx, row in enumerate(predicted_titles[:50]):
    print(test_data['text'][idx], "\n", "Згенерована назва:", row, '\n\n')

Generated 100 titles

Generated 200 titles

Generated 300 titles

Generated 400 titles

Generated 500 titles

<|endoftext|>Текст: Глава правління Укрпошти Ігор Смілянський сказав в інтерв'ю проекту KRYM, що його місячний оклад збільшився в 2018 році і становить 748 тис. грн. "Місячний оклад - 748 тис. грн. Більшість наших співробітників вважає, що я повинен отримувати 40 тис. Це 10 зарплат листонош", - зазначив він. Смілянський нагадав, що у нього три вищі освіти в американських вузах, і підкреслив, що може відзвітувати за всі свої доходи, на відміну від критикуючих його за високу зарплату депутатів, які "все життя в політиці, але у них будинок в Конча-Заспі і Mercedes". "Я розумію джерело цього. Я знаю, що більшості людей простіше сприймати депутата, який офіційно отримує $300, живе в Конча-Заспі в триповерховому будинку і їздить на Mercedes... Вони знають, що він вкрав, але скільки точно, не знають, а моя зарплата відома", - додав глава Укрпошти. За словами Смілянського, після його п

In [33]:
# виділення справжніх заголовків, що були зібрані разом з текстами новин
orig_titles = []
for idx, row in enumerate(test_data['text']):
    title = row.split("\n Назва: ")[1].replace("<|endoftext|>", "")
    orig_titles.append(title)
    if idx % 10 == 0:
      print(title)

Глава Укрпошти: Моя зарплата - 748 тис. грн. Більшість співробітників вважає, що я повинен отримувати 40 тис.
"Айтішники"-шахраї вкрали у банку майже 1,5 млн грн під виглядом перевірки платіжної системи
"Ощадбанк" хоче відсудити у Росії 15 млрд грн через анексію Криму
«Не в захваті». Маск розкритикував Гейтса після придбання Porsche
Кабмін опублікував протокол зустрічі ЄС, України та Росії з газового транзиту
Обама наказав надати Україні військову допомогу на $20 млн
"Азов" діє в межах закону, ситуація контрольована, - Шкіряк
Роналду заборонив мамі наживо дивитися свої матчі
В Києві зареєстрували новий випадок ботулізму
Сьогодні обстановка в зоні АТО залишалася спокійною, - прес-центр
У Словенії згоріла дерев’яна скульптура Трампа
Стало відомо, хто замінить сестер Семеренко на етапі Кубка світу в Поклюці
Аграрний комітет розглянув всі поправки до законопроєкту про ринок землі
Дуда підписав закон про спрощення залучення іноземних медиків на роботу до Польщі
3G з’явиться у Києві вже цьог

In [34]:
len(predicted_titles), len(orig_titles)

(500, 500)

In [35]:
for idx, pd in enumerate(predicted_titles):
  if idx % 10 == 0:
    print(pd)

Смілянський розповів, що його зарплата зросла в 2018 році на $500 тис. і він може відзвітувати за всі доходи, - ЗМІ
Правоохоронці викрили шахраїв, які ошукали банк на 1,4 млн грн, - МВС. ФОТОрепортаж
Яценюк вимагає від Росії відшкодувати збитки через втрату майна на українській території, - Кабмін
«Не в захваті від розмов з Гейтсом»: засновник Tesla розкритикував Гейтса за рішення купити Tesla — ЗМІ
У Кабміні підписали протокол про транзит російського газу через Україну до 2034 року — Оржель про переговори з Газпромом щодо транзиту газу до Європи — НВ Бізнес
США нададуть Україні $20 млн на оборону, - Білий дім США надали Україні $20 млн на оборону, - Білий дім США нададуть Україні $20 млн на оборону, - Білий дім США нададуть $20 млн на оборону, - Білий дім
У МВС не виключають провокацій під Радою через візит "Азову" до парламенту, - Шкіряк
Роналду заборонив своїй мамі дивитися матчі Манчестер Юнайтед наживо через проблеми зі здоров'ям, - ЗМІ
У Києві від ботулізму померла 9-річна дівчин

In [36]:
# збереження згенерованих заголовків
with open("/kaggle/working/generated_by_gpt.json", "w", encoding='utf-8') as fw:
    json.dump(predicted_titles, fw)

In [63]:
with open("/kaggle/working/generated_by_gpt.json", "r", encoding='utf-8') as fr:
    predicted_titles = json.load(fr)[:400]
for idx, pd in enumerate(predicted_titles):
  if idx % 10 == 0:
    print(pd)

Смілянський розповів, що його зарплата зросла в 2018 році на $500 тис. і він може відзвітувати за всі доходи, - ЗМІ
Правоохоронці викрили шахраїв, які ошукали банк на 1,4 млн грн, - МВС. ФОТОрепортаж
Яценюк вимагає від Росії відшкодувати збитки через втрату майна на українській території, - Кабмін
«Не в захваті від розмов з Гейтсом»: засновник Tesla розкритикував Гейтса за рішення купити Tesla — ЗМІ
У Кабміні підписали протокол про транзит російського газу через Україну до 2034 року — Оржель про переговори з Газпромом щодо транзиту газу до Європи — НВ Бізнес
США нададуть Україні $20 млн на оборону, - Білий дім США надали Україні $20 млн на оборону, - Білий дім США нададуть Україні $20 млн на оборону, - Білий дім США нададуть $20 млн на оборону, - Білий дім
У МВС не виключають провокацій під Радою через візит "Азову" до парламенту, - Шкіряк
Роналду заборонив своїй мамі дивитися матчі Манчестер Юнайтед наживо через проблеми зі здоров'ям, - ЗМІ
У Києві від ботулізму померла 9-річна дівчин

### 3.1 Оцінка точності моделі

Задачі генерації контенту мають одне суттєве обмеження - для оцінки ефективності побудованих рішень необхідно встановлювати власні метрики та проводити оцінювання з залученням людей.
Якщо для задач класифікації простір правильних відповідей обмежено, то для значної частини задач генерації він майже неперервний, обмежений лише словником мови.

Ручне оцінювання потребує багато ресурсів, тому у цій роботі буде використано один з методів автоматичної оцінки.

#### BERTScore

Метод автоматичної оцінки BERTScore використовує попередньо натреновані векторні представлення моделей на базі архітектури BERT і зіставляє слова у текстах-передбаченнях з правильними за допомогою косинусної подібності. Було продемонстровано, що цей метод корелює з людською оцінкою на рівні речення.

In [ ]:
# оцінка заголовків, що були згенеровані натренованою моделлю
#results = bertscore.compute(predictions=predicted_titles, references=orig_titles, model_type="microsoft/mdeberta-v3-base")
#{metric: np.mean(results[metric]) for metric in ["precision", "recall", "f1"]}
# причина: https://github.com/huggingface/evaluate/issues/739
from bert_score import BERTScorer

scorer = BERTScorer(model_type="microsoft/mdeberta-v3-base")
scorer._tokenizer.model_max_length = 512
P, R, F1 = scorer.score(cands=predicted_titles, refs=orig_titles[:400], batch_size=16)
results = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}
results

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from di

{'precision': 0.6218864917755127,
 'recall': 0.7000886797904968,
 'f1': 0.6571658849716187}

## 4. Тестування великої мовної моделі за допомогою API

У цьому розділі Ви порівняєте результати генерації натренованого рішення з дійсно великою мовною моделлю, що має десятки мільярдів параметрів.
Для цього необхідно зареєструватися на платформі [groq.com](https://groq.com), що надає тарифікований доступ до загальнодоступних мовних моделей через API. Після реєстрації новим користувачам виділяється безкоштовний ліміт запитів для певних моделей.

Детальніше про безкоштовний доступ за [посиланням](https://community.groq.com/t/is-there-a-free-tier-and-what-are-its-limits/790).

Документація платформи доступна за [посиланням](https://console.groq.com/docs/quickstart).

In [61]:
# скопіюйте свій ключ доступу з https://console.groq.com/keys та вставте його нижче
client = Groq(api_key="YOUR_API_KEY")

In [69]:
def generate_response(inputs, model, prompt, postprocess=True, max_tokens=512, temperature=0.7, sample_logs=25):
    outputs = []
    for idx, row in enumerate(inputs):
        if (idx+1) % 100 == 0:
            print(f"Processed {idx+1} texts")
        # видалення заголовку та спеціальних токенів з вхідної послідовності
        row = row.split("\n Назва:")[0].replace("<|endoftext|>", "")
        message = prompt.format(text=row)
        if not sample_logs or idx % sample_logs == 0:
          print(message)
        # взаємодія з моделями відбувається у форматі чату
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": message}],
            temperature=temperature,
            top_p=0.95,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
            stop=None,
            stream=False
        )
        output = response.choices[0].message.content
        if postprocess:
            output = output.replace("*", "")  # може використовуватися моделями для стилізації відповіді
            if output.startswith("Назва:"):
                output = output[6:]
        outputs.append(output.strip())
        if not sample_logs or idx % sample_logs == 0:
          print(output, "\n")
        time.sleep(1)  # безкоштовний доступ обмежує кількість запитів на хвилину
    return outputs

#### Варіанти для генерації заголовків

Варіант = Номер_у_списку % 4 + 1

|**№**| Temperature |
|-----|-------------|
|**1**| 0.5         |
|**2**| 0.6         |
|**3**| 0.7         |
|**4**| 0.8         |

In [70]:
# генерація заголовків для тестового набору моделлю Qwen 3.8, що містить 27 мільярдів параметрів
model = "qwen/qwen3.8-27b"
prompt = "Підбери назву до наведеного нижче тексту. Надай лише назву тексту після ключового слова 'Назва:' без будь-яких коментарів.\n{text}\nНазва:"

qwen_titles = generate_response(test_data["text"][:400], model, prompt)

Підбери назву до наведеного нижче тексту. Надай лише назву тексту після ключового слова 'Назва:' без будь-яких коментарів.
Текст: Глава правління Укрпошти Ігор Смілянський сказав в інтерв'ю проекту KRYM, що його місячний оклад збільшився в 2018 році і становить 748 тис. грн. "Місячний оклад - 748 тис. грн. Більшість наших співробітників вважає, що я повинен отримувати 40 тис. Це 10 зарплат листонош", - зазначив він. Смілянський нагадав, що у нього три вищі освіти в американських вузах, і підкреслив, що може відзвітувати за всі свої доходи, на відміну від критикуючих його за високу зарплату депутатів, які "все життя в політиці, але у них будинок в Конча-Заспі і Mercedes". "Я розумію джерело цього. Я знаю, що більшості людей простіше сприймати депутата, який офіційно отримує $300, живе в Конча-Заспі в триповерховому будинку і їздить на Mercedes... Вони знають, що він вкрав, але скільки точно, не знають, а моя зарплата відома", - додав глава Укрпошти. За словами Смілянського, після його п

In [71]:
print(len(qwen_titles))
for idx, qt in enumerate(qwen_titles):
  if idx % 10 == 0:
    print(qt)

400
Глава Укрпошти розкрив розмір своєї зарплати
Кіберполіція розкрила шахрайство на 1,4 млн грн у банку
Яценюк заявив, що Ощадбанк має намір відсудити у Росії 15 млрд грн
Ілон Маск висловив незадоволення від розмов з Біллом Гейтсом
Протокол щодо транзиту російського газу через Україну підписано
США виділять $20 млн на військову допомогу Україні
Радник глави МВС заявив про мирний хід санкціонованої акції біля Верховної Ради
Роналду заборонив мамі дивитися його матчі через проблеми зі здоров'ям
В'ялена риба викликала ботулізм у киянина, а вакцина від отрути в лікарні відсутня
Затишшя на фронті дозволило доставити гуманітарну допомогу мешканцям Донбасу
Пожежа знищила статую Дональда Трампа в Словенії
Лєсніков розповів про ротацію у складі жіночої збірної України на етапах Кубка світу
Депутати погодили остаточну версію законопроєкту про ринок землі
Польща спрощує умови працевлаштування іноземних лікарів для боротьби з COVID-19
Київстар запускає 3G-мережу в Києві з 16 червня
Протестувальни

In [72]:
# збереження згенерованих заголовків
with open("generated_by_qwen.json", "w", encoding='utf-8') as fw:
    json.dump(qwen_titles, fw)

In [73]:
with open("generated_by_qwen.json", "r", encoding='utf-8') as fr:
    titles_generated_by_qwen = json.load(fr)
print(len(titles_generated_by_qwen))

for idx, tq in enumerate(titles_generated_by_qwen):
  if idx % 10 == 0:
    print(tq)

400
Глава Укрпошти розкрив розмір своєї зарплати
Кіберполіція розкрила шахрайство на 1,4 млн грн у банку
Яценюк заявив, що Ощадбанк має намір відсудити у Росії 15 млрд грн
Ілон Маск висловив незадоволення від розмов з Біллом Гейтсом
Протокол щодо транзиту російського газу через Україну підписано
США виділять $20 млн на військову допомогу Україні
Радник глави МВС заявив про мирний хід санкціонованої акції біля Верховної Ради
Роналду заборонив мамі дивитися його матчі через проблеми зі здоров'ям
В'ялена риба викликала ботулізм у киянина, а вакцина від отрути в лікарні відсутня
Затишшя на фронті дозволило доставити гуманітарну допомогу мешканцям Донбасу
Пожежа знищила статую Дональда Трампа в Словенії
Лєсніков розповів про ротацію у складі жіночої збірної України на етапах Кубка світу
Депутати погодили остаточну версію законопроєкту про ринок землі
Польща спрощує умови працевлаштування іноземних лікарів для боротьби з COVID-19
Київстар запускає 3G-мережу в Києві з 16 червня
Протестувальни

In [74]:
# оцінка заголовків, що були згенеровані моделлю Qwen3.8 27b
#results = bertscore.compute(predictions=qwen_titles, references=orig_titles, model_type="microsoft/mdeberta-v3-base")
#{metric: np.mean(results[metric]) for metric in ["precision", "recall", "f1"]}

P, R, F1 = scorer.score(cands=qwen_titles, refs=orig_titles[:400], batch_size=16)
results = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}
results

{'precision': 0.6894235014915466,
 'recall': 0.6808689832687378,
 'f1': 0.683579683303833}

## 5.* Спробуйте покращити якість генерацій для моделі з попереднього розділу